## Test Individual Location

Test with incomplete 2025 data

In [2]:
import os, ssl, io, json, pytz, numpy as np, pandas as pd, requests
from datetime import datetime
from meteostat import Stations, Hourly
import calendar
from pandas.errors import EmptyDataError

# -----------------------------
# Helper functions
# -----------------------------

def safe_date(year, month, day):
    last_day = calendar.monthrange(year, month)[1]
    return datetime(year, month, min(day, last_day))

def convert_utc_to_local(df, local_tz):
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.reset_index(level='station', drop=True)
        df.index = pd.to_datetime(df.index)
    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    return df.tz_convert(local_tz)

def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    if timezone:
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        df.index = df.index.tz_localize(None)
    return df.loc[(df.index >= start_date) & (df.index <= end_date)]

def get_time_shift(timezone_name):
    timezone = pytz.timezone(timezone_name)
    now = datetime.now(timezone)
    return int(now.utcoffset().total_seconds() // 3600)

def calculate_hdd_cdd(df, temperature_column):
    df[temperature_column + '_F'] = df[temperature_column] * 9 / 5 + 32
    base_temperature = 65
    df['date'] = df.index.to_series().dt.date
    daily_mean_temp = df.groupby('date')[temperature_column + '_F'].mean()
    hdd = (base_temperature - daily_mean_temp).clip(lower=0).sum()
    cdd = (daily_mean_temp - base_temperature).clip(lower=0).sum()
    return int(hdd), int(cdd)

def get_parameters_MERRA2(lat, lon, year, end_month, end_day):
    url = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}{end_month}{end_day}&format=EPW"
    print("Fetching MERRA2 from NASA POWER API...")
    response = requests.get(url)
    csv_data = io.StringIO(response.text)
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    if calendar.isleap(int(year)):
        df = df.iloc[:8784]
    else:
        df = df.iloc[:8760]
    return df

# -----------------------------
# Main function
# -----------------------------

def get_data(lat, lon, year, save_folder, save_name, end_month, end_day):
    os.makedirs(save_folder, exist_ok=True)
    print(f"Save folder: {save_folder}")

    ssl._create_default_https_context = ssl._create_unverified_context
    start = datetime(year - 1, 12, 31)
    end = safe_date(year, int(end_month), int(end_day))

    stations = Stations().nearby(lat, lon)
    station_number = 0
    incomplete_timeseries = True

    # Find NOAA data
    while incomplete_timeseries:
        station_number += 1
        station_info = stations.fetch(station_number)
        wmo = str(station_info.index.values[-1])
        if os.path.exists(f'{save_folder}/{wmo}_{year}.epw'):
            print(f"EPW already exists for WMO {wmo}, skipping...")
            return ("done", True, {}, 0, 0, wmo, "", "", True)

        data = Hourly(station_info, start, end, model=True).fetch()
        if (len(data.index) > 100) & (station_number > 1):
            data = data.loc[data.index.get_level_values('station').unique()[-1]]

        if len(data.index) > 80:
            incomplete_timeseries = False

    timezone = station_info['timezone'].values[-1]
    elevation = station_info['elevation'].values[-1]
    wmo = str(station_info.index.values[-1])
    station_name = station_info['name'].values[-1]
    state = station_info['region'].values[-1]
    country = station_info['country'].values[-1]
    lat_station = station_info['latitude'].values[-1]
    lon_station = station_info['longitude'].values[-1]

    # Process NOAA data
    data_local = filter_dataframe_by_date(
        convert_utc_to_local(data, timezone),
        datetime(year, 1, 1),
        datetime(year, int(end_month), int(end_day))
    )
    data_hourly = data_local.resample("H").mean().interpolate(method="linear", limit=3, limit_direction="both")
    hdd, cdd = calculate_hdd_cdd(data_hourly, "temp")
    print(f"HDD: {hdd}, CDD: {cdd}")

    data_hourly.to_csv(f"{save_folder}/{save_name}_NOAA.csv")

    # Improved MERRA2 block with retries
    attempt = 0
    while True:
        attempt += 1
        print(f"Fetching MERRA2 from NASA POWER API... attempt {attempt}")
        try:
            df_merra2 = get_parameters_MERRA2(lat, lon, year, end_month, end_day)
            print("✅ Successfully fetched MERRA2 data.")
            break
        except (EmptyDataError, pd.errors.ParserError) as e:
            print(f"Attempt {attempt} failed due to data error: {e}")
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed due to network error: {e}")




    df_merra2.to_csv(f"{save_folder}/{save_name}_MERRA2.csv")
    print("Saved MERRA2 data.")

    # Prepare structured return
    info_dict = {
        "timeshift": get_time_shift(timezone),
        "elevation": elevation,
        "wmo": wmo,
        "station_name": station_name,
        "state": state,
        "country": country,
        "lat": lat_station,
        "lon": lon_station,
        "weather_file_type": "AMY"
    }

    result = ("done", True, info_dict, hdd, cdd, wmo, lat_station, lon_station, False)

    # JSON-style clean print
    output_json = {
        "status": "done",
        "success": True,
        "metadata": info_dict,
        "HDD": hdd,
        "CDD": cdd,
        "WMO": wmo,
        "station_lat": lat_station,
        "station_lon": lon_station,
        "epw_exists": False
    }
    print("===== SUMMARY JSON =====")
    print(json.dumps(output_json, indent=2))

    return result

# -----------------------------
# Run
# -----------------------------

year = 2025
end_month = "06"
end_day = "30"
lat = 40.121676
lon = -111.639805

save_folder = 'Landan_2025'
save_name = 'SpanishFork_2025'

get_data(lat, lon, year, save_folder, save_name, end_month, end_day)


Save folder: Landan_2025
HDD: 3274, CDD: 277
Fetching MERRA2 from NASA POWER API... attempt 1
Fetching MERRA2 from NASA POWER API...
Attempt 1 failed due to data error: No columns to parse from file
Fetching MERRA2 from NASA POWER API... attempt 2
Fetching MERRA2 from NASA POWER API...
✅ Successfully fetched MERRA2 data.
Saved MERRA2 data.
===== SUMMARY JSON =====
{
  "status": "done",
  "success": true,
  "metadata": {
    "timeshift": -6,
    "elevation": 1380.0,
    "wmo": "JJZBL",
    "station_name": "Spanish Fork Municipal Airport",
    "state": "UT",
    "country": "US",
    "lat": 40.145,
    "lon": -111.6677,
    "weather_file_type": "AMY"
  },
  "HDD": 3274,
  "CDD": 277,
  "WMO": "JJZBL",
  "station_lat": 40.145,
  "station_lon": -111.6677,
  "epw_exists": false
}


('done',
 True,
 {'timeshift': -6,
  'elevation': 1380.0,
  'wmo': 'JJZBL',
  'station_name': 'Spanish Fork Municipal Airport',
  'state': 'UT',
  'country': 'US',
  'lat': 40.145,
  'lon': -111.6677,
  'weather_file_type': 'AMY'},
 3274,
 277,
 'JJZBL',
 40.145,
 -111.6677,
 False)